<a href="https://colab.research.google.com/github/fre-denni/genAIforUX-research/blob/main/logbook_data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install spacy google-genai pandas
!python -m spacy download en_core_web_sm

In [ ]:
import os
import pandas as pd
from google import genai
from google.genai import types
import torch
import json
import re

from IPython.display import clear_output
clear_output()
print("Environment setup correctly!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#folder variables, change as you wish
main_folder = "logbook analysis"

# Define the base folder
base_folder = f"/content/drive/MyDrive/{main_folder}/"

# Define input and output folders
input_folder = os.path.join(base_folder, "logbook")
output_folder = os.path.join(base_folder, "output")
output_images_folder = os.path.join(base_folder, "output_images")

# Create output folders if they don't exist
os.makedirs(output_folder, exist_ok=True)
os.makedirs(output_images_folder, exist_ok=True)

print(f"Base folder: {base_folder}")
print(f"Input folder set to: {input_folder}")
print(f"Output folder created/verified: {output_folder}")
print(f"Output images folder created/verified: {output_images_folder}")


In [ ]:

from google.colab import userdata
from google import genai

NOME_SECRET = 'GEMINI_API'

try:
    # Recupera la chiave
    api_key = userdata.get(NOME_SECRET)

    if not api_key:
        print(f"ERRORE: La chiave non è stata trovata. Controlla che il nome sia esattamente '{NOME_SECRET}' e che la levetta 'Notebook access' sia attiva.")
    else:
        # Stampiamo inizio e fine per scovare eventuali spazi vuoti o virgolette
        print(f"🔍 Controllo formato: La chiave inizia con '{api_key[:5]}...' e finisce con '...{api_key[-5:]}'")

        # Inizializza il client
        client = genai.Client(api_key=api_key)
        print("✅ Client inizializzato in locale.")

        # Facciamo una vera chiamata di test per validare la chiave sul server
        print("⏳ Provo a contattare Google AI Studio...")
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents='Rispondi solo con "API funzionante al 100%!"'
        )
        print(f"🤖 Risposta da Gemini: {response.text}")

except userdata.SecretNotFoundError:
    print(f"❌ ERRORE CRITICO: Il secret chiamato '{NOME_SECRET}' non esiste nei tuoi Segreti di Colab.")
except Exception as e:
    print(f"❌ ERRORE API (Chiave invalida o problemi di rete): {e}")

In [ ]:
# Force the config of output as json
json_config = types.GenerateContentConfig(
    response_mime_type="application/json",
    temperature=0.0 # more deterministic outputs
)

In [ ]:
nlp = spacy.load("en_core_web_sm") # English language
ruler = nlp.add_pipe("entity_ruler", before="ner")

# Add custom rules for design methodologies
patterns = [
    # A1 - User Journey Maps & varianti
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "user"}, {"LOWER": "journey"}, {"LOWER": {"IN": ["map", "maps"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "ujm"}]},

    # A1 -Storyboards
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": {"IN": ["storyboard", "storyboards"]}}]},

    # A2 - Mental Model Diagrams (cattura sia "mental model" che "mental model diagram")
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "mental"}, {"LOWER": "model"}, {"LOWER": {"IN": ["diagram", "diagrams"]}, "OP": "?"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "mmd"}]},

    # A2 -Mind & Empathy Maps
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "mind"}, {"LOWER": {"IN": ["map", "maps"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "empathy"}, {"LOWER": {"IN": ["map", "maps"]}}]},

    # A1 - User Jobs & JTBD
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "user"}, {"LOWER": {"IN": ["job", "jobs"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "jobs"}, {"LOWER": "to"}, {"LOWER": "be"}, {"LOWER": "done"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "jtbd"}]},

    # A1 - User Stories
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "user"}, {"LOWER": {"IN": ["story", "stories"]}}]},

    # A1 - Task Analysis (cattura sia "task analysis" che "task analysis diagram")
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "task"}, {"LOWER": "analysis"}, {"LOWER": {"IN": ["diagram", "diagrams"]}, "OP": "?"}]},

    # A2 - Concept / Conceptual Maps
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": {"IN": ["concept", "conceptual"]}}, {"LOWER": {"IN": ["map", "maps"]}}]},

    # A3 - Research
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "literature"}, {"LOWER": "review"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "benchmarking"}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": {"IN": ["persona", "personas"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "digital"}, {"LOWER": "ethnography"}]},

    # A4 - Mapping (System, Service, Stakeholder) - cattura map, maps e mapping
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "system"}, {"LOWER": {"IN": ["map", "maps", "mapping"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "service"}, {"LOWER": {"IN": ["map", "maps", "mapping"]}}]},
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "stakeholder"}, {"LOWER": {"IN": ["map", "maps", "mapping"]}}]},

    # A2 - Affinity Diagram
    {"label": "DESIGN METHOD", "pattern": [{"LOWER": "affinity"}, {"LOWER": {"IN": ["diagram", "diagrams"]}}]}
]
ruler.add_patterns(patterns)

ruler.add_patterns(patterns)
print("Added custom config to spacy")

In [ ]:
nlp = spacy.load("en_core_web_sm")

def anonymizer(student_id, df_students):
  replacements ={
      "Margherita Pillan": "[PROFESSOR]",
      "Margherita": "[PROFESSOR]",
      "Pillan": "[PROFESSOR]",
      "Federico Denni": "[ASSISTANT]",
      "Federico": "[ASSISTANT]",
      "Denni": "[ASSISTANT]"
  }

  student_id = str(student_id)
  student_row = df_students[df_students['Codice persona'].astype(str) == student_id]

  if not student_row.empty:
    full_name = student_row.iloc[0]['Cognome-Nome'].title()
    parts = full_name.split()
    if len(parts) >= 2:
      cognome = parts[0]
      nome = " ".join(parts[1:])

      #all possible combinations
      replacements[f"{nome} {cognome}"] = student_id
      replacements[f"{cognome} {nome}"] = student_id
      replacements[nome] = student_id
      replacements[cognome] = student_id


  # Order the replacemetns
  sorted_replacements = dict(sorted(replacements.items(), key=lambda item: len(item[0]), reverse=True))


  def anonymize_text(text):
    if not isinstance(text, str) or not text.strip():
      return ""

    # Aimed subsitution
    for name, replacement in sorted_replacements.items():
      # Usa regex con word boundaries per non tagliare pezzi di altre parole
      pattern = r'\b' + re.escape(name) + r'\b'
      text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)

    # Generic subsition
    doc = nlp(text)
    cleaned_text = text
    persons = sorted([ent.text for ent in doc.ents if ent.label_ == "PERSON"], key=len, reverse=True)

    for person in persons:
        if person != student_id and "[PROFESSOR]" not in person and "[ASSISTANT]" not in person:
           cleaned_text = cleaned_text.replace(person, "[STUDENT/PERSON]")

    return cleaned_text

  return anonymize_text


In [ ]:
# filtering prompt
text_clean_prompt = """You are an expert UX teaching assistant validating extracted text.

CONTEXT:
Assignment: {assignment}
Topic: {topic}
Supervisor Reasoning: {reasoning}
Detected Methods: {design_methodologies}
Detected Entities: {entities}

TEXT:
---
{text}
---

EVALUATION CRITERIA:
Assess if the text explicitly discusses:
1. A UX Design Methodology (Application or Explanation).
2. A Digital Tool (Figma, Miro, Python, etc.).
3. Reflections (Course, Tools, Methodology, AI).
4. AI Usage.

DECISION RULE:
If the text discusses AT LEAST ONE of the above criteria OR contains highly relevant design methods/entities detected by spaCy, set "keep" to true.
If the text DOES NOT discuss these criteria AND lacks relevant entities/methodologies (e.g., it's just an index, generic filler, or unreadable OCR), set "keep" to false.

Return ONLY a JSON:
{{
  "keep": true/false,
  "category": "Method Application" | "Method Explanation" | "Tool Usage" | "Reflection" | "Irrelevant",
  "reasoning": "Brief reason..."
}}
"""

In [ ]:
from tqdm.auto import tqdm
tqdm.pandas(desc="Elaborazione LLM")

students_path = os.path.join(base_folder,"students-team-ux-grade.csv")
df_students = pd.read_csv(students_path)

df_logbook = pd.read_csv(os.path.join(base_folder, "extractions_results.csv"))

print("Cleaning phase 1: Anonimity")
def apply_anonymizer(row):
  student_id = str(row['ID'])
  anon_func = anonymizer(student_id, df_students)

  #apply anonimization on text
  text_content = str(row['Text'])
  text = anon_func(text_content) if pd.notnull(row['Text']) and text_content.strip() != "" else ""
  return pd.Series(text)

df_logbook['Text'] = df_logbook.apply(apply_anonymizer, axis=1)

print("Cleaning phase 2: Images")
def check_image_exists(row):
  #skip no images
  if row['Layout'] != 'Image_Scheme':
    return True

  path = str(row['Path_Image'])
  #return true only if image present
  return os.path.exists(path)

df_logbook = df_logbook[df_logbook.apply(check_image_exists, axis=1)]

print("Cleaning phase 3: Entity extraction")
def extract_entities_and_methods(text):
  if not isinstance(text, str) or not text.strip():
    return "", ""

  doc = nlp(text)
  methods = set([ent.text for ent in doc.ents if ent.label_ == "DESIGN METHOD"])
  others = set([ent.text for ent in doc.ents if ent.label_ != "DESIGN METHOD"])

  return ", ".join(methods), ", ".join(others)

#Temporary column
df_logbook['full'] = df_logbook['Text'].fillna('') + " \n " + df_logbook['OCR_Text'].fillna('')

#Estraiamo e inseriamo nelle due nuove colonne
df_logbook[['entities', 'design_methodologies']] = df_logbook['full'].apply(
    lambda x : pd.Series(extract_entities_and_methods(x))
)

print("Cleaning phase 4: Validation and Classification of text")
def evaluate_text_with_llm(row):
  text_to_eval = row['full'].strip()

  # Discard text that is without entities
  if not text_to_eval and not row['entities'] and not row['design_methodologies']:
     return False

  prompt = text_clean_prompt.format(
      assignment=row.get('Assignment', 'Unknown'),
      topic=row.get('Context_Topic', 'Unknown'),
      reasoning=row.get('Reasoning', 'None'),
      design_methodologies=row.get('design_methodologies', 'None'),
      entities=row.get('entities', 'None'),
      text=text_to_eval
  )

  try:
    response = client.models.generate_content(
          model='gemini-2.5-flash',
          contents=prompt,
          config=json_config
    )

    # Pulizia della risposta markdown JSON
    raw_text = response.text.strip()
    if raw_text.startswith("```json"):
        raw_text = raw_text[7:]
    if raw_text.endswith("```"):
        raw_text = raw_text[:-3]

    result = json.loads(raw_text.strip())

    # Facciamo ritornare solo il booleano per decidere se tenere la riga
    return result.get("keep", False)

  except Exception as e:
    print(f"Gemini API Error for row page {row['Page']}: {e}")
    return True

# Usiamo progress_apply per vedere lo stato d'avanzamento, visto che chiamare le API richiederà qualche minuto
df_logbook['keep_flag'] = df_logbook.progress_apply(evaluate_text_with_llm, axis=1)

# Eliminiamo tutte le righe flaggate per l'eliminazione
df_cleaned = df_logbook[df_logbook['keep_flag'] == True].copy()

# Rimuoviamo le colonne temporanee usate per il processamento
df_cleaned = df_cleaned.drop(columns=['full', 'keep_flag'])

# Salviamo il risultato finale
os.makedirs(base_folder, exist_ok=True)
df_cleaned.to_csv(os.path.join(base_folder, "extractions_cleaned.csv"), index=False)

print("Operation completed!")
print(f"Original rows_ {len(df_logbook)}")
print(f"Remained rows: {len(df_cleaned)}")

display(df_cleaned.head())